# 📊 Notebook 02 – Exploratory Data Analysis (EDA)
**Goal:** understand patterns in the tweets: sentiment balance, airline differences, complaint reasons, tweet length and the most common words.

> EDA uses the **raw** file because we want to see the data as it really is. Cleaning happens in Notebook 03.

## 1. Import Libraries and Load Data

In [ ]:
# Import libraries for analysis and visualization
import re
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

try:
    import plotly.express as px          # interactive charts (optional)
    PLOTLY_OK = True
except ImportError:
    PLOTLY_OK = False
    print("plotly not installed -> interactive chart will be skipped (pip install plotly)")

try:
    from wordcloud import WordCloud      # word clouds (optional)
    WORDCLOUD_OK = True
except ImportError:
    WORDCLOUD_OK = False
    print("wordcloud not installed -> word clouds will be skipped (pip install wordcloud)")

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

In [ ]:
# Set up project paths (works on your D: drive path, or falls back to the parent of 02_Notebooks)
from pathlib import Path

BASE_DIR = Path(r"D:\DataScience\notebooks\SentimentIQ")
if not BASE_DIR.exists():
    BASE_DIR = Path.cwd().parent          # fallback: notebook is running inside 02_Notebooks/

RAW_PATH       = BASE_DIR / "01_Data" / "Raw" / "Tweets.csv"
PROCESSED_DIR  = BASE_DIR / "01_Data" / "Processed"
CLEANED_PATH   = PROCESSED_DIR / "cleaned_tweets.csv"
MODELS_DIR     = BASE_DIR / "03_Models"
REPORTS_DIR    = BASE_DIR / "05_Reports"

for folder in (PROCESSED_DIR, MODELS_DIR, REPORTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42      # same seed everywhere -> reproducible results
TEST_SIZE    = 0.20    # 80% training / 20% testing

print("Project folder :", BASE_DIR)
print("Raw data exists:", RAW_PATH.exists())

In [ ]:
# Load the raw dataset and define a fixed colour for each sentiment (used in every chart)
try:
    df = pd.read_csv(RAW_PATH)
except FileNotFoundError:
    raise FileNotFoundError(f"Tweets.csv not found at {RAW_PATH}")

ORDER   = ["positive", "neutral", "negative"]
PALETTE = {"positive": "#2ecc71", "neutral": "#f5b041", "negative": "#e74c3c"}
print("Rows:", len(df))

## 2. Sentiment Distribution

In [ ]:
# Count how many tweets belong to each sentiment
sentiment_counts = df["airline_sentiment"].value_counts().reindex(ORDER)
sentiment_counts.to_frame("tweet_count")

## 3. Sentiment Count Visualization

In [ ]:
# Bar chart of sentiment counts with the number written on top of each bar
ax = sns.countplot(data=df, x="airline_sentiment", order=ORDER, palette=PALETTE)
for bar in ax.patches:
    ax.annotate(f"{int(bar.get_height()):,}", (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                ha="center", va="bottom", fontsize=11)
ax.set_title("Sentiment Distribution")
ax.set_xlabel("Sentiment")
ax.set_ylabel("Number of tweets")
plt.show()

## 4. Sentiment Percentage

In [ ]:
# Convert counts into percentages
sentiment_pct = (df["airline_sentiment"].value_counts(normalize=True).reindex(ORDER) * 100).round(2)
sentiment_pct.to_frame("percentage (%)")

## 5. Pie Chart of Sentiment Distribution

In [ ]:
# Pie chart showing each sentiment's share
plt.figure(figsize=(6, 6))
plt.pie(sentiment_counts, labels=[s.title() for s in ORDER], autopct="%1.1f%%", startangle=90,
        colors=[PALETTE[s] for s in ORDER], wedgeprops={"edgecolor": "white"})
plt.title("Share of Each Sentiment")
plt.show()

**Observation:** the data is clearly **imbalanced**: negative tweets are roughly 6 in 10. A model could look "accurate" just by guessing *negative*, so later we will judge models using **precision, recall and macro F1** – not accuracy alone.

## 6. Airline-wise Sentiment Analysis

In [ ]:
# Number of tweets of each sentiment for every airline
airline_ct = pd.crosstab(df["airline"], df["airline_sentiment"])[ORDER]
airline_ct["total"] = airline_ct.sum(axis=1)
airline_ct.sort_values("total", ascending=False)

In [ ]:
# Grouped bar chart: sentiment counts per airline
plt.figure(figsize=(11, 5))
sns.countplot(data=df, x="airline", hue="airline_sentiment", hue_order=ORDER, palette=PALETTE)
plt.title("Sentiment by Airline")
plt.xlabel("Airline")
plt.ylabel("Number of tweets")
plt.legend(title="Sentiment")
plt.show()

In [ ]:
# 100% stacked bar: percentage of each sentiment within every airline (fair comparison)
airline_pct = pd.crosstab(df["airline"], df["airline_sentiment"], normalize="index")[ORDER] * 100

if PLOTLY_OK:
    long_df = airline_pct.reset_index().melt(id_vars="airline", var_name="sentiment", value_name="percent")
    fig = px.bar(long_df, y="airline", x="percent", color="sentiment", orientation="h",
                 color_discrete_map=PALETTE, category_orders={"sentiment": ORDER},
                 title="Sentiment Share (%) per Airline")
    fig.show()
else:
    airline_pct.plot(kind="barh", stacked=True, color=[PALETTE[s] for s in ORDER], figsize=(10, 5))
    plt.title("Sentiment Share (%) per Airline")
    plt.xlabel("Percent")
    plt.show()

## 7. Negative Reason Analysis

In [ ]:
# Why are customers unhappy? Count the reasons given for negative tweets
neg_reasons = df.loc[df["airline_sentiment"] == "negative", "negativereason"].value_counts()
neg_reasons.to_frame("count")

In [ ]:
# Horizontal bar chart of negative reasons
plt.figure(figsize=(9, 5))
sns.barplot(x=neg_reasons.values, y=neg_reasons.index, color="#e74c3c")
plt.title("Reasons for Negative Tweets")
plt.xlabel("Number of tweets")
plt.ylabel("")
plt.show()

In [ ]:
# Heatmap: which airline suffers from which complaint (row-wise percentage)
reason_airline = pd.crosstab(df["airline"], df["negativereason"], normalize="index") * 100
plt.figure(figsize=(13, 5))
sns.heatmap(reason_airline, annot=True, fmt=".0f", cmap="Reds", cbar_kws={"label": "% of negative tweets"})
plt.title("Negative Reasons by Airline (%)")
plt.show()

> ⚠️ `negativereason` exists **only** for negative tweets, so using it as a feature would leak the answer. We use it for analysis only, never for training.

## 8. Tweet Length Calculation

In [ ]:
# Add character count and word count columns
df["char_count"] = df["text"].str.len()
df["word_count"] = df["text"].str.split().str.len()
df[["char_count", "word_count"]].describe().round(1)

## 9. Tweet Length Distribution

In [ ]:
# Histogram of tweet length in characters and words
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df["char_count"], bins=40, kde=True, color="#3498db", ax=axes[0])
axes[0].set_title("Characters per Tweet")
sns.histplot(df["word_count"], bins=30, kde=True, color="#9b59b6", ax=axes[1])
axes[1].set_title("Words per Tweet")
plt.tight_layout()
plt.show()

## 10. Average Tweet Length by Sentiment

In [ ]:
# Average length for each sentiment, plus a box plot to show the spread
avg_len = df.groupby("airline_sentiment")[["char_count", "word_count"]].mean().reindex(ORDER).round(1)
display(avg_len)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=avg_len.reset_index(), x="airline_sentiment", y="char_count", palette=PALETTE, ax=axes[0])
axes[0].set_title("Average Characters per Tweet")
sns.boxplot(data=df, x="airline_sentiment", y="word_count", order=ORDER, palette=PALETTE, ax=axes[1])
axes[1].set_title("Word Count Spread")
plt.tight_layout()
plt.show()

## 11. Most Frequent Words

In [ ]:
# Light cleaning just for word counting (the full preprocessing comes in Notebook 03)
EXTRA_STOP = {"amp", "flight", "flights", "co", "http", "https", "rt"}
STOP = set(ENGLISH_STOP_WORDS) | EXTRA_STOP

def light_clean(text):
    text = text.lower()
    text = re.sub(r"http\S+|@\w+", " ", text)      # remove URLs and @mentions
    text = re.sub(r"[^a-z\s]", " ", text)          # keep letters only
    return [w for w in text.split() if w not in STOP and len(w) > 2]

df["tokens"] = df["text"].apply(light_clean)
all_words = Counter(w for toks in df["tokens"] for w in toks)
top_words = pd.DataFrame(all_words.most_common(20), columns=["word", "count"])
top_words

In [ ]:
# Bar chart of the 20 most frequent words
plt.figure(figsize=(9, 6))
sns.barplot(data=top_words, x="count", y="word", color="#3498db")
plt.title("Top 20 Most Frequent Words")
plt.show()

In [ ]:
# Top 10 words for each sentiment side by side
top_by_sentiment = {}
for s in ORDER:
    counter = Counter(w for toks in df.loc[df["airline_sentiment"] == s, "tokens"] for w in toks)
    top_by_sentiment[s] = [f"{w} ({n})" for w, n in counter.most_common(10)]
pd.DataFrame(top_by_sentiment, index=range(1, 11))

## 12–14. Word Clouds

In [ ]:
# Helper that draws a word cloud for one sentiment
def show_wordcloud(sentiment, colormap):
    if not WORDCLOUD_OK:
        print("wordcloud library is missing -> pip install wordcloud")
        return
    words = " ".join(w for toks in df.loc[df["airline_sentiment"] == sentiment, "tokens"] for w in toks)
    if not words:
        print("No words available for", sentiment)
        return
    cloud = WordCloud(width=900, height=450, background_color="white", colormap=colormap,
                      max_words=120, random_state=42).generate(words)
    plt.figure(figsize=(11, 5))
    plt.imshow(cloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"{sentiment.title()} Tweets – Word Cloud", fontsize=14)
    plt.show()

### Positive Sentiment Word Cloud

In [ ]:
# Word cloud for positive tweets
show_wordcloud("positive", "Greens")

### Negative Sentiment Word Cloud

In [ ]:
# Word cloud for negative tweets
show_wordcloud("negative", "Reds")

### Neutral Sentiment Word Cloud

In [ ]:
# Word cloud for neutral tweets
show_wordcloud("neutral", "Oranges")

## ✅ Observations (compare with your own charts)
1. **Class imbalance:** ~63% negative, ~21% neutral, ~16% positive → use macro F1 and class-aware evaluation.
2. **Airlines:** United, US Airways and American receive the most tweets and are mostly negative; Virgin America has the most balanced sentiment.
3. **Complaints:** the biggest negative reasons are typically *Customer Service Issue*, *Late Flight* and *Can't Tell*.
4. **Length:** tweets are short (Twitter limit), and negative tweets tend to be slightly longer – people explain problems.
5. **Words:** positive clouds show *thank, great, love, best*; negative clouds show *delay, cancelled, hour, customer service, wait*; neutral tweets are mostly questions and information requests.

**Next:** Notebook 03 – Cleaning and Preprocessing.